### **1. Database Architecture & Schema Design**
Tahap awal ini berfokus pada perancangan skema relasional untuk sistem manajemen data *TechCorp*. Implementasi mencakup:
- Inisialisasi koneksi SQLite dan kursor eksekusi.
- Definisi struktur tabel (`DDL`) untuk entitas **Departments**, **Employees**, dan **Projects**.
- Penerapan *Referential Integrity* melalui *Foreign Keys* untuk menjamin konsistensi hubungan antar entitas.

In [ ]:
import os
import sqlite3
import pandas as pd

DB_PATH = os.getenv("TECHCORP_NOTEBOOK_DB", "techcorp_demo.sqlite")
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")  # Enforce SQLite referential integrity
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS Projects")
cursor.execute("DROP TABLE IF EXISTS Employees")
cursor.execute("DROP TABLE IF EXISTS Departments")

cursor.execute("""
CREATE TABLE Departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL
);
""")

cursor.execute("""
CREATE TABLE Employees (
    employee_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT UNIQUE,
    hire_date TEXT,
    department_id INTEGER,
    FOREIGN KEY (department_id) REFERENCES Departments(department_id)
);
""")

cursor.execute("""
CREATE TABLE Projects (
    project_id INTEGER PRIMARY KEY,
    project_name TEXT NOT NULL,
    start_date TEXT,
    end_date TEXT,
    budget REAL,
    department_id INTEGER,
    FOREIGN KEY (department_id) REFERENCES Departments(department_id)
);
""")

conn.commit()
print("Database Schema Created Successfully!")

### **2. Data Ingestion Simulation**
Proses populasi database dengan dataset simulasi untuk memodelkan lingkungan operasional perusahaan. Data dimasukkan secara *batch* menggunakan `executemany` untuk efisiensi, mencakup variasi data departemen, profil karyawan, dan detail proyek.

In [ ]:
departments_data = [
    (1, 'Human Resources'),
    (2, 'IT'),
    (3, 'Marketing'),
    (4, 'Finance'),
    (5, 'Sales')
]
cursor.executemany("INSERT INTO Departments VALUES (?, ?)", departments_data)

employees_data = [
    (101, 'John', 'Doe', 'john.doe@techcorp.com', '2022-01-15', 2),
    (102, 'Jane', 'Smith', 'jane.smith@techcorp.com', '2021-05-20', 1),
    (103, 'Emily', 'Johnson', 'emily.j@techcorp.com', '2023-03-10', 3),
    (104, 'Michael', 'Brown', 'michael.b@techcorp.com', '2020-11-01', 2),
    (105, 'Sarah', 'Davis', 'sarah.davis@techcorp.com', '2022-07-23', 4),
    (106, 'David', 'Wilson', 'david.wilson@techcorp.com', '2021-09-14', 5),
    (107, 'Laura', 'Martinez', 'laura.m@techcorp.com', '2023-01-05', 2),
    (108, 'James', 'Taylor', 'james.t@techcorp.com', '2020-04-18', 1),
    (109, 'Olivia', 'Anderson', 'olivia.a@techcorp.com', '2022-12-01', 3),
    (110, 'Robert', 'Thomas', 'robert.t@techcorp.com', '2021-06-30', 4)
]
cursor.executemany("INSERT INTO Employees VALUES (?, ?, ?, ?, ?, ?)", employees_data)

projects_data = [
    (201, 'Website Redesign', '2023-01-01', '2023-06-30', 50000, 2),
    (202, 'Employee Onboarding', '2023-02-15', '2023-04-30', 10000, 1),
    (203, 'Market Research', '2023-03-01', '2023-08-31', 25000, 3),
    (204, 'Financial Audit', '2023-04-01', '2023-05-31', 15000, 4),
    (205, 'Sales Strategy', '2023-05-01', '2023-11-30', 30000, 5),
    (206, 'App Development', '2023-06-01', '2023-12-31', 80000, 2)
]
cursor.executemany("INSERT INTO Projects VALUES (?, ?, ?, ?, ?, ?)", projects_data)

conn.commit()
print("Data Inserted Successfully into TechCorp SQLite database!")

### **3. Data Integrity & Relational Mapping Validation**
Verifikasi data pasca-ingesti untuk memastikan integritas relasi antar tabel. Menggunakan `JOIN` query untuk memvalidasi bahwa setiap entitas karyawan telah terasosiasi dengan departemen yang tepat sebelum dilakukan analisis lebih lanjut.

In [ ]:
query_check = """
SELECT 
    e.first_name || ' ' || e.last_name AS Full_Name,
    e.email,
    d.department_name
FROM Employees e
JOIN Departments d ON e.department_id = d.department_id
ORDER BY d.department_name;
"""

df_view = pd.read_sql(query_check, conn)
display(df_view)

### **4. Financial Analysis: Departmental Budget Allocation**
Analisis agregasi untuk memetakan distribusi anggaran proyek di seluruh unit bisnis. Visualisasi data menggunakan *Bar Plot* (Seaborn) diimplementasikan untuk memberikan *insight* komparatif yang cepat mengenai alokasi sumber daya finansial per departemen.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

query_budget = """
SELECT 
    d.department_name,
    COUNT(p.project_id) as total_projects,
    SUM(p.budget) as total_budget
FROM Projects p
JOIN Departments d ON p.department_id = d.department_id
GROUP BY d.department_name
ORDER BY total_budget DESC;
"""

df_budget = pd.read_sql(query_budget, conn)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_budget, x='total_budget', y='department_name', hue='department_name', palette='Blues_r', legend=False)
plt.title('Total Budget Project per Departemen', fontsize=14, fontweight='bold')
plt.xlabel('Budget (USD)')
plt.ylabel('Departemen')
plt.show()

display(df_budget)

### **5. Targeted Data Extraction**
Ekstraksi data spesifik untuk kebutuhan operasional:
- **Talent Mapping:** Mengidentifikasi personil dalam divisi teknis (IT) untuk alokasi sumber daya.
- **High-Value Project Identification:** Menaring proyek strategis dengan valuasi anggaran >$25,000 untuk prioritas monitoring manajemen.

In [ ]:
print("--- 1. Mencari Karyawan di Departemen IT (Dept ID = 2) ---")
query_it = """
SELECT first_name, last_name, email 
FROM Employees 
WHERE department_id = 2;
"""
display(pd.read_sql(query_it, conn))

print("\n--- 2. Mencari Proyek dengan Budget di atas $25,000 ---")
query_high_budget = """
SELECT project_name, budget 
FROM Projects 
WHERE budget > 25000 
ORDER BY budget DESC;
"""
display(pd.read_sql(query_high_budget, conn))

### **6. Workforce Distribution & Project Timeline Metrics**
Analisis operasional lanjutan mencakup:
- **Headcount Analytics:** Mengukur densitas karyawan per departemen untuk evaluasi beban kerja.
- **Duration Analysis:** Kalkulasi durasi proyek (dalam hari) berbasis data temporal (`start_date` vs `end_date`) untuk estimasi *turnaround time* proyek.

In [ ]:
print("--- 3. Menghitung Jumlah Karyawan per Departemen ---")
query_count_emp = """
SELECT 
    d.department_name,
    COUNT(e.employee_id) as total_employees
FROM Employees e
JOIN Departments d ON e.department_id = d.department_id
GROUP BY d.department_name
ORDER BY total_employees DESC;
"""
display(pd.read_sql(query_count_emp, conn))

print("\n--- 4. Rata-rata Durasi Project (Estimasi Kasar) ---")
query_duration = """
SELECT 
    project_name,
    (julianday(end_date) - julianday(start_date)) as duration_days
FROM Projects
ORDER BY duration_days DESC;
"""
display(pd.read_sql(query_duration, conn))

### **7. Data Lifecycle Management (CRUD Operations)**
Simulasi pemeliharaan data dinamis untuk mencerminkan skenario bisnis nyata:
- **Budget Adjustment:** Revisi anggaran proyek 'Website Redesign' merespons perubahan kebutuhan bisnis (*UPDATE*).
- **Personnel Offboarding:** Penghapusan data karyawan non-aktif dari sistem database (*DELETE*) serta validasi penghapusan untuk keamanan data.

In [ ]:
print("--- SEBELUM UPDATE: Cek Budget Website Redesign ---")
display(pd.read_sql("SELECT project_name, budget FROM Projects WHERE project_id = 201", conn))

cursor.execute("""
UPDATE Projects 
SET budget = 60000 
WHERE project_id = 201;
""")
conn.commit()
print("Budget berhasil di-update!")

print("--- SETELAH UPDATE: ---")
display(pd.read_sql("SELECT project_name, budget FROM Projects WHERE project_id = 201", conn))

print("\n--- Menghapus Data Karyawan ID 110 (Robert Thomas) ---")
cursor.execute("DELETE FROM Employees WHERE employee_id = 110;")
conn.commit()


check_emp = pd.read_sql("SELECT * FROM Employees WHERE employee_id = 110", conn)
if check_emp.empty:
    print(" Data karyawan ID 110 berhasil dihapus dari database.")
else:
    print(" Data masih ada.")

### **8. Executive Reporting: Project Governance**
Penyusunan laporan komprehensif yang mengintegrasikan data lintas entitas. Query ini menyajikan pandangan holistik mengenai Proyek, Anggaran, dan *Person in Charge (PIC)*—dengan asumsi logika bisnis bahwa karyawan senior (tenure < 2021) bertindak sebagai manajer proyek.

In [ ]:
query_report = """
SELECT 
    d.department_name,
    e.first_name || ' ' || e.last_name as manager_name,
    p.project_name,
    p.budget
FROM Departments d
JOIN Employees e ON d.department_id = e.department_id
JOIN Projects p ON d.department_id = p.department_id
WHERE e.hire_date < '2021-01-01' -- Anggap karyawan lama adalah manager/lead
ORDER BY d.department_name;
"""

print("--- Laporan Proyek & PIC (Senior Staff) ---")
display(pd.read_sql(query_report, conn))

### **9. Resource Cleanup**
Menutup koneksi SQLite secara bersih setelah seluruh query analitik dan pelaporan selesai dijalankan.

In [ ]:
conn.close()
print("SQLite database connection closed cleanly.")